# Visualize LitePT Flat CSV Inference

This notebook visualizes outputs from `scripts/run_litept_flat_csv_inference.py`.

It treats `semantic_mask.npy` and `confidence.npy` as flat point arrays: index `i` corresponds to row `i` in the source CSV. It does not reshape predictions into a fake range image. The views below use CSV geometry: `Cxd/Cyd` for camera projection, `azimuth/vertical` for angular view, and `x/y` for LiDAR top view.

In [ ]:
from __future__ import annotations

import json
import math
import xml.etree.ElementTree as ET
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

plt.rcParams["figure.figsize"] = (16, 7)
plt.rcParams["axes.grid"] = False

PROJECT_ROOT = Path("/home/a60116606/git_repo/noise_seg/pipline_v0")

# Output produced by scripts/run_litept_flat_csv_inference.py
INFERENCE_DIR = PROJECT_ROOT / "output" / "HL320_output_sam3_manual_104" / "litept_flat_csv_inference"

# Original flat CSV point clouds used for inference.
CSV_DIR = PROJECT_ROOT / "data" / "HL320_0622" / "2026-06-04_17-04-22_0000_LIDAR2_1090_1245" / "csv_shift_3_1090_1245"

# Optional: original images. Leave as None if you only want point plots.
IMAGE_DIR = PROJECT_ROOT / "data" / "HL320_0622" / "2026-06-04_17-04-22_0000_LIDAR2_1090_1245" / "img2"

# Optional: labels.xml from point_labeler gives nicer class names/colors.
LABELS_XML = Path("/home/a60116606/git_repo/point_labeler/HL320_output_sam3_manual_104/labels.xml")

# Change this to inspect another frame.
FRAME_ID = "000091"

MANIFEST_PATH = INFERENCE_DIR / "litept_flat_csv_inference_manifest.json"
print("INFERENCE_DIR:", INFERENCE_DIR)
print("CSV_DIR:", CSV_DIR)
print("IMAGE_DIR:", IMAGE_DIR)
print("FRAME_ID:", FRAME_ID)

In [ ]:
def read_json(path):
    if not path or not Path(path).is_file():
        return {}
    return json.loads(Path(path).read_text(encoding="utf-8"))


def read_labels_xml(path):
    path = Path(path)
    if not path.is_file():
        return {}, {}
    root = ET.parse(path).getroot()
    id_to_name = {}
    id_to_color = {}
    for label in root.findall("label"):
        raw_id = (label.findtext("id") or "").strip()
        name = (label.findtext("name") or "").strip()
        if not raw_id or not name:
            continue
        label_id = int(raw_id)
        id_to_name[label_id] = name
        raw_color = (label.findtext("color") or "").strip()
        if raw_color:
            values = [int(float(token)) for token in raw_color.replace(",", " ").split()[:3]]
            if len(values) == 3:
                id_to_color[label_id] = np.asarray(values, dtype=np.uint8)
    return id_to_name, id_to_color


def metadata_path_for_frame(frame_id):
    return INFERENCE_DIR / frame_id / "metadata.json"


def build_class_lookup(frame_id):
    manifest = read_json(MANIFEST_PATH)
    metadata = read_json(metadata_path_for_frame(frame_id))
    id_to_name, id_to_color = read_labels_xml(LABELS_XML)

    for payload in (manifest, metadata):
        names = payload.get("class_names") or []
        source_ids = payload.get("training_id_to_source_id") or list(range(len(names)))
        for name, source_id in zip(names, source_ids):
            id_to_name.setdefault(int(source_id), str(name))
        semantic_classes = payload.get("semantic_classes") or {}
        for name, source_id in semantic_classes.items():
            id_to_name.setdefault(int(source_id), str(name))

    id_to_name.setdefault(0, "background")
    id_to_name.setdefault(255, "ignore")
    id_to_color.setdefault(0, np.array([35, 35, 35], dtype=np.uint8))
    id_to_color.setdefault(255, np.array([125, 125, 125], dtype=np.uint8))
    return id_to_name, id_to_color, manifest, metadata


id_to_name, id_to_color, inference_manifest, frame_metadata = build_class_lookup(FRAME_ID)
print("manifest exists:", MANIFEST_PATH.is_file())
print("frame metadata exists:", metadata_path_for_frame(FRAME_ID).is_file())
print("classes known:", len(id_to_name))
print("manifest frames:", len(inference_manifest.get("frames", [])))

In [ ]:
IMAGE_SUFFIXES = (".jpg", ".jpeg", ".png", ".bmp", ".webp")


def split_table_row(line):
    if "," in line:
        return [value.strip() for value in line.split(",")]
    return line.replace("\t", " ").split()


def load_csv_columns(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"CSV does not exist: {path}")
    with path.open("r", encoding="utf-8") as handle:
        header = None
        for line in handle:
            if line.strip():
                header = split_table_row(line.strip())
                break
        if header is None:
            raise ValueError(f"CSV is empty: {path}")
        column_map = {name.strip().casefold(): index for index, name in enumerate(header)}
        wanted = ["x", "y", "z", "intensity", "cxd", "cyd", "azimuth", "vertical", "slot", "pixel", "hcell", "vcell"]
        values = {name: [] for name in wanted if name in column_map}
        max_idx = max((column_map[name] for name in values), default=-1)
        for line_number, line in enumerate(handle, start=2):
            if not line.strip():
                continue
            tokens = split_table_row(line.strip())
            if len(tokens) <= max_idx:
                continue
            for name in values:
                try:
                    values[name].append(float(tokens[column_map[name]]))
                except ValueError:
                    values[name].append(np.nan)
    return {name: np.asarray(items, dtype=np.float64) for name, items in values.items()}


def csv_path_for_frame(frame_id):
    metadata = read_json(metadata_path_for_frame(frame_id))
    raw = metadata.get("source_csv")
    if raw and Path(raw).is_file():
        return Path(raw)
    return CSV_DIR / f"{frame_id}.csv"


def image_path_for_frame(frame_id):
    if IMAGE_DIR is None:
        return None
    for suffix in IMAGE_SUFFIXES:
        candidate = Path(IMAGE_DIR) / f"{frame_id}{suffix}"
        if candidate.is_file():
            return candidate
    return None


def load_image(path):
    if path is None or not Path(path).is_file():
        return None
    from PIL import Image
    return np.asarray(Image.open(path).convert("RGB"), dtype=np.uint8)


def points_from_csv(csv_cols):
    if not {"x", "y", "z"}.issubset(csv_cols):
        return None
    return np.stack([csv_cols["x"], csv_cols["y"], csv_cols["z"]], axis=1).astype(np.float32)

In [ ]:
def class_name(label_id):
    return id_to_name.get(int(label_id), f"id_{int(label_id)}")


def stable_color(label_id):
    label_id = int(label_id)
    if label_id in id_to_color:
        return id_to_color[label_id]
    rng = np.random.default_rng((label_id * 1009 + 17) & 0xFFFFFFFF)
    return rng.integers(40, 240, size=3, dtype=np.uint8)


def colors_for_labels(labels):
    flat = np.asarray(labels).reshape(-1)
    colors = np.zeros((flat.size, 3), dtype=np.uint8)
    for label_id in np.unique(flat):
        colors[flat == label_id] = stable_color(int(label_id))
    return colors


def print_distribution(labels, confidence=None, max_rows=80):
    flat = np.asarray(labels).reshape(-1)
    conf = None if confidence is None else np.asarray(confidence).reshape(-1)
    ids, counts = np.unique(flat, return_counts=True)
    pairs = sorted([(int(i), int(c)) for i, c in zip(ids, counts)], key=lambda item: item[1], reverse=True)
    rows = ["| class id | name | points | percent | mean confidence |", "|---:|---|---:|---:|---:|"]
    for label_id, count in pairs[:max_rows]:
        if conf is None:
            mean_conf = ""
        else:
            values = conf[flat == label_id]
            mean_conf = f"{float(np.nanmean(values)):.4f}" if values.size else ""
        rows.append(f"| {label_id} | {class_name(label_id)} | {count} | {100.0 * count / flat.size:.3f}% | {mean_conf} |")
    display(Markdown("\n".join(rows)))


def sample_indices(n, max_points=300_000):
    idx = np.arange(n)
    if max_points is not None and n > max_points:
        rng = np.random.default_rng(42)
        idx = np.sort(rng.choice(idx, size=max_points, replace=False))
    return idx


def align_to_shortest(labels, confidence, csv_cols):
    lengths = [int(np.asarray(labels).size), int(np.asarray(confidence).size)]
    lengths += [int(values.size) for values in csv_cols.values()]
    target = min(lengths)
    if len(set(lengths)) != 1:
        print("WARNING: length mismatch detected:", {"labels": labels.size, "confidence": confidence.size, **{k: v.size for k, v in csv_cols.items()}})
        print("Diagnostic plots will use first", target, "items only.")
    labels = np.asarray(labels).reshape(-1)[:target]
    confidence = np.asarray(confidence).reshape(-1)[:target]
    csv_cols = {key: np.asarray(values).reshape(-1)[:target] for key, values in csv_cols.items()}
    return labels, confidence, csv_cols

In [ ]:
def load_inference_frame(frame_id):
    frame_dir = INFERENCE_DIR / frame_id
    semantic_path = frame_dir / "semantic_mask.npy"
    confidence_path = frame_dir / "confidence.npy"
    csv_path = csv_path_for_frame(frame_id)
    image_path = image_path_for_frame(frame_id)

    labels = np.load(semantic_path, allow_pickle=False).reshape(-1)
    if confidence_path.is_file():
        confidence = np.load(confidence_path, allow_pickle=False).reshape(-1).astype(np.float32)
    else:
        confidence = np.zeros(labels.shape, dtype=np.float32)
    csv_cols = load_csv_columns(csv_path)
    labels, confidence, csv_cols = align_to_shortest(labels, confidence, csv_cols)
    points = points_from_csv(csv_cols)
    image = load_image(image_path)
    return {
        "frame_id": frame_id,
        "frame_dir": frame_dir,
        "semantic_path": semantic_path,
        "confidence_path": confidence_path,
        "csv_path": csv_path,
        "image_path": image_path,
        "labels": labels,
        "confidence": confidence,
        "csv_cols": csv_cols,
        "points": points,
        "image": image,
        "metadata": read_json(frame_dir / "metadata.json"),
    }


frame = load_inference_frame(FRAME_ID)
print("frame:", frame["frame_id"])
print("semantic:", frame["semantic_path"], frame["labels"].shape, frame["labels"].dtype)
print("confidence:", frame["confidence_path"], frame["confidence"].shape, frame["confidence"].dtype)
print("csv:", frame["csv_path"], {k: v.shape for k, v in frame["csv_cols"].items()})
print("image:", frame["image_path"], None if frame["image"] is None else frame["image"].shape)
print("confidence min/mean/max:", float(np.nanmin(frame["confidence"])), float(np.nanmean(frame["confidence"])), float(np.nanmax(frame["confidence"])))
print_distribution(frame["labels"], frame["confidence"])

In [ ]:
def projection_confidence_canvas(cxd, cyd, confidence, image):
    if image is not None:
        height, width = image.shape[:2]
    else:
        finite = np.isfinite(cxd) & np.isfinite(cyd)
        width = int(np.ceil(np.nanmax(cxd[finite]))) + 1 if np.any(finite) else 1
        height = int(np.ceil(np.nanmax(cyd[finite]))) + 1 if np.any(finite) else 1
    canvas = np.full((height, width), np.nan, dtype=np.float32)
    valid = np.isfinite(cxd) & np.isfinite(cyd) & np.isfinite(confidence)
    u = np.rint(cxd[valid]).astype(np.int64)
    v = np.rint(cyd[valid]).astype(np.int64)
    inside = (u >= 0) & (u < width) & (v >= 0) & (v < height)
    if np.any(inside):
        # Multiple points may land on the same pixel; keep the strongest confidence.
        np.maximum.at(canvas, (v[inside], u[inside]), confidence[valid][inside])
    return canvas


def visualize_inference_frame(frame_id, max_points=300_000, point_size=0.7, invert_angular_y=False):
    frame = load_inference_frame(frame_id)
    labels = frame["labels"]
    confidence = frame["confidence"]
    csv_cols = frame["csv_cols"]
    points = frame["points"]
    image = frame["image"]
    idx = sample_indices(labels.size, max_points=max_points)
    colors = colors_for_labels(labels[idx]).astype(np.float32) / 255.0

    fig, axes = plt.subplots(2, 3, figsize=(24, 13))
    axes = axes.reshape(-1)

    if {"cxd", "cyd"}.issubset(csv_cols):
        cxd = csv_cols["cxd"][idx]
        cyd = csv_cols["cyd"][idx]
        finite = np.isfinite(cxd) & np.isfinite(cyd)
        if image is not None:
            axes[0].imshow(image)
            axes[0].set_xlim(0, image.shape[1])
            axes[0].set_ylim(image.shape[0], 0)
        else:
            axes[0].invert_yaxis()
        axes[0].scatter(cxd[finite], cyd[finite], c=colors[finite], s=point_size, linewidths=0, alpha=0.88)
        axes[0].set_title("Semantic labels: camera projection Cxd/Cyd")
        axes[0].set_xlabel("Cxd")
        axes[0].set_ylabel("Cyd")

        conf_scatter = axes[3].scatter(
            cxd[finite],
            cyd[finite],
            c=confidence[idx][finite],
            s=point_size,
            linewidths=0,
            alpha=0.9,
            cmap="viridis",
            vmin=0.0,
            vmax=1.0,
        )
        if image is not None:
            axes[3].imshow(image, alpha=0.35)
            axes[3].set_xlim(0, image.shape[1])
            axes[3].set_ylim(image.shape[0], 0)
        else:
            axes[3].invert_yaxis()
        axes[3].set_title("Confidence scatter: camera projection")
        axes[3].set_xlabel("Cxd")
        axes[3].set_ylabel("Cyd")
        fig.colorbar(conf_scatter, ax=axes[3], fraction=0.046, pad=0.04)

        canvas = projection_confidence_canvas(csv_cols["cxd"], csv_cols["cyd"], confidence, image)
        axes[5].imshow(canvas, cmap="viridis", vmin=0.0, vmax=1.0, interpolation="nearest")
        axes[5].set_title("Confidence map via rounded Cxd/Cyd")
        axes[5].axis("off")
    else:
        axes[0].text(0.5, 0.5, "CSV Cxd/Cyd missing", ha="center", va="center")
        axes[3].text(0.5, 0.5, "CSV Cxd/Cyd missing", ha="center", va="center")
        axes[5].text(0.5, 0.5, "CSV Cxd/Cyd missing", ha="center", va="center")

    if {"azimuth", "vertical"}.issubset(csv_cols):
        x_ang = -1.0 * csv_cols["azimuth"][idx]
        y_ang = -1.0 * csv_cols["vertical"][idx]
        finite = np.isfinite(x_ang) & np.isfinite(y_ang)
        axes[1].scatter(x_ang[finite], y_ang[finite], c=colors[finite], s=point_size, linewidths=0)
        axes[1].set_title("Semantic labels: angular view -azimuth / -vertical")
        axes[1].set_xlabel("-azimuth")
        axes[1].set_ylabel("-vertical")
        if invert_angular_y:
            axes[1].invert_yaxis()

        conf_ang = axes[4].scatter(
            x_ang[finite],
            y_ang[finite],
            c=confidence[idx][finite],
            s=point_size,
            linewidths=0,
            cmap="viridis",
            vmin=0.0,
            vmax=1.0,
        )
        axes[4].set_title("Confidence: angular view")
        axes[4].set_xlabel("-azimuth")
        axes[4].set_ylabel("-vertical")
        if invert_angular_y:
            axes[4].invert_yaxis()
        fig.colorbar(conf_ang, ax=axes[4], fraction=0.046, pad=0.04)
    else:
        axes[1].text(0.5, 0.5, "CSV azimuth/vertical missing", ha="center", va="center")
        axes[4].text(0.5, 0.5, "CSV azimuth/vertical missing", ha="center", va="center")

    if points is not None and points.shape[0] == labels.size:
        pts = points[idx]
        axes[2].scatter(pts[:, 0], pts[:, 1], c=colors, s=point_size, linewidths=0)
        axes[2].set_title("Semantic labels: LiDAR XY top view")
        axes[2].set_xlabel("x")
        axes[2].set_ylabel("y")
        axes[2].axis("equal")
    else:
        axes[2].text(0.5, 0.5, "CSV x/y/z missing or count mismatch", ha="center", va="center")

    plt.suptitle(f"LitePT flat CSV inference: {frame_id}, {labels.size} points", y=1.02)
    plt.tight_layout()
    plt.show()


visualize_inference_frame(FRAME_ID, point_size=0.75)

In [ ]:
# Browse several inference frames.
frame_dirs = sorted(path for path in INFERENCE_DIR.iterdir() if path.is_dir())
frame_ids = [path.name for path in frame_dirs]
print("frames:", len(frame_ids), frame_ids[:10])

start = 0
count = 5
for frame_id in frame_ids[start : start + count]:
    visualize_inference_frame(frame_id, max_points=150_000, point_size=0.45)

In [ ]:
def save_frame_preview(frame_id, out_dir=None, point_size=0.6):
    frame = load_inference_frame(frame_id)
    labels = frame["labels"]
    confidence = frame["confidence"]
    csv_cols = frame["csv_cols"]
    image = frame["image"]
    if not {"cxd", "cyd"}.issubset(csv_cols):
        raise ValueError(f"No Cxd/Cyd columns for {frame_id}")

    colors = colors_for_labels(labels).astype(np.float32) / 255.0
    cxd = csv_cols["cxd"]
    cyd = csv_cols["cyd"]
    finite = np.isfinite(cxd) & np.isfinite(cyd)

    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    if image is not None:
        axes[0].imshow(image)
        axes[0].set_xlim(0, image.shape[1])
        axes[0].set_ylim(image.shape[0], 0)
    else:
        axes[0].invert_yaxis()
    axes[0].scatter(cxd[finite], cyd[finite], c=colors[finite], s=point_size, linewidths=0, alpha=0.9)
    axes[0].set_title(f"{frame_id}: semantic labels")
    axes[0].axis("off")

    canvas = projection_confidence_canvas(cxd, cyd, confidence, image)
    axes[1].imshow(canvas, cmap="viridis", vmin=0.0, vmax=1.0, interpolation="nearest")
    axes[1].set_title(f"{frame_id}: confidence map")
    axes[1].axis("off")

    out_dir = Path(out_dir) if out_dir is not None else INFERENCE_DIR / "previews"
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{frame_id}_litept_flat_csv_preview.png"
    fig.savefig(out_path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return out_path


# Uncomment to save previews for all frames.
# saved = [save_frame_preview(frame_id) for frame_id in frame_ids]
# print(f"saved {len(saved)} previews to {saved[0].parent if saved else ''}")